# Portfolio API — Zerodha (`slug: zerodha`)

Exercises all `/portfolio/*` endpoints scoped to the `zerodha` source.

**Auth pre-req:** Log in to [kite.zerodha.com](https://kite.zerodha.com) inside the AlphaForge Chrome session (`--remote-debugging-port=9299`). Set `ZERODHA_USER_ID` in `backend/.env.cred.local`.

In [ ]:
import json
from pathlib import Path

SLUG     = "zerodha"
MODE     = "http"          # "in_process" | "http"
BASE     = "http://localhost:8000/api/v1"
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""

def get(path, **kw):
    r = client.get(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

def post(path, **kw):
    r = client.post(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

print(f"Mode: {MODE}  slug: {SLUG}")

## 1. Source info

`status: ready` when `ZERODHA_USER_ID` is set, `unconfigured` otherwise.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Triggers CDP login → enctoken → Kite OMS holdings fetch. Enctoken cached in `.cache/brokers/zerodha.json`.

> Requires `MODE="http"` with a live server and an open Chrome session.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:5]:
    print(f"  {h['symbol']:14} qty={h['quantity']:<6}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Upload CSV (offline fallback)

Skips Chrome — upload a CSV from console.zerodha.com directly.

In [ ]:
csv_path = FIXTURES / "zerodha_holdings.csv"
if csv_path.exists():
    with csv_path.open("rb") as f:
        r = client.post(
            f"{PREFIX}/portfolio/sources/{SLUG}/upload",
            files={"file": (csv_path.name, f, "text/csv")},
        )
    print(r.status_code)
    body = r.json()
    print(f"Uploaded {body.get('holdings_count')} holdings")
else:
    print(f"No fixture at {csv_path} — drop a Zerodha CSV export there.")

## 4. Holdings — zerodha only

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    print(f"  {h['symbol']:14} qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 5. Allocation (zerodha)

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 6. Treemap (zerodha)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 7. Rebalance (zerodha)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 8. Reset zerodha cache

In [ ]:
from app.modules.brokers import SOURCES

SOURCES[SLUG].reset()
status, body = get(f"/portfolio/sources/{SLUG}")
print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")